# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

## My Capstone Plan

**Domain:** Unified Student Engagement Ecosystem -- Education and Financing Choices

**User:** Students (Class 10/12 pass-outs, undergraduates) searching for college options, course guidance, scholarships, and education loan information in India.

**Success looks like:** Agent answers 85%+ of student queries faithfully from the knowledge base. For out-of-scope queries (e.g., stock prices), it clearly says it does not know. Memory persists within a session -- a student can say 'my name is Priya' and later the agent refers to her by name.

**Tool I will add:** Web search (DuckDuckGo) -- for live scholarship deadlines, current admission dates, and news that the static KB cannot cover.

**Deployment choice:** Streamlit UI -- browser-based chat interface students can use on any device.


---
## 0. Setup

In [1]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# ============================================================
# !pip install langgraph langchain-groq langchain-core chromadb \
#              sentence-transformers ragas ddgs python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph:    {version('langgraph')}")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM:          ✅ {r.content}")

ModuleNotFoundError: No module named 'chromadb'

---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain. Write them as strings or load from files.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain (don't repeat the same topic)
- Documents should be specific enough to answer concrete questions

In [ ]:
# Domain documents -- Student Engagement Ecosystem (Education + Financing)
# 12 documents covering college selection, admissions, loans, and scholarships

DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "Engineering College Selection Criteria",
        "text": """Choosing the right engineering college in India requires evaluating several key factors.
NIRF Ranking published by the Ministry of Education is the most reliable ranking system -- prefer
colleges in the top 200 for better placement outcomes. Accreditation by NBA (National Board of
Accreditation) indicates that individual programs meet quality benchmarks, while NAAC accreditation
evaluates the overall institution. Check the Student-to-Faculty Ratio: a ratio below 20:1 is
considered good. Placement statistics -- look for average package, highest package, and the
percentage of students placed. Government colleges (NITs, IIITs, State Engineering colleges)
typically charge Rs 50,000 to Rs 2,00,000 per year in fees; private autonomous colleges range from
Rs 1,00,000 to Rs 3,00,000 per year. Lateral entry is available in most states for Diploma holders
directly into the second year of B.Tech. Campus location matters for internships -- colleges near
tech hubs like Hyderabad, Bengaluru, Pune, and Chennai offer more industry interaction."""
    },
    {
        "id": "doc_002",
        "topic": "JEE Main and JEE Advanced Admission Process",
        "text": """JEE Main is the national entrance exam conducted by NTA (National Testing Agency) for admission
to NITs, IIITs, and Government Funded Technical Institutions (GFTIs). JEE Main is held twice a year
-- January session and April session. Students can appear in both and the best score is considered.
Eligibility: passed or appearing in Class 12 with Physics, Chemistry, and Mathematics. There is no
age limit from 2021 onwards. JEE Main score is used for the JoSAA counselling process for NITs,
IIITs, and GFTIs. JEE Advanced is the gateway to IITs -- only the top 2,50,000 qualifiers in JEE
Main are eligible to appear. JEE Advanced is conducted by one of the seven IIT zones on rotation.
IIT admissions happen through JoSAA counselling based on JEE Advanced rank. For state engineering
colleges, separate state-level exams like TS EAMCET, AP EAMCET, MHT-CET, KCET, and WBJEE are
conducted. Preparation timeline: begin from Class 11 for JEE -- focus on NCERT first, then coaching
material. Mock tests and previous year papers are essential from October of Class 12."""
    },
    {
        "id": "doc_003",
        "topic": "NEET UG Medical Admission Process",
        "text": """NEET UG (National Eligibility cum Entrance Test Undergraduate) is the single national entrance
exam for MBBS, BDS, AYUSH, and Nursing admissions across India, conducted by NTA. Eligibility:
minimum 50% in Class 12 PCB (Physics, Chemistry, Biology) for general category; 40% for SC/ST/OBC.
Age: 17 years minimum on December 31 of the admission year. NEET has 180 questions (45 each from
Physics, Chemistry, Botany, Zoology), total 720 marks, duration 3 hours 20 minutes. Marking: +4
correct, -1 wrong. Top scorers in NEET qualify for 15% All India Quota (AIQ) seats counselled by
MCC (Medical Counselling Committee). Remaining 85% of government seats are filled by state
counselling. Private medical college fees range from Rs 5 lakh to Rs 25 lakh per year. Government
MBBS seats cost Rs 10,000 to Rs 50,000 per year. Cut-off for government MBBS varies yearly --
approximately 600+ marks for general category AIQ seats."""
    },
    {
        "id": "doc_004",
        "topic": "Central Sector Scholarship Scheme for Undergraduate Students",
        "text": """The Central Sector Scheme of Scholarships for College and University Students (CSSS) is funded
by the Ministry of Education, Government of India. Eligibility: students who scored above 80th
percentile in their Class 12 board exam and have annual family income below Rs 8 lakh. The
scholarship provides Rs 10,000 per year for the first three years of undergraduate study and
Rs 20,000 per year for professional courses in the 4th and 5th year. Renewal requires minimum 50%
marks in the previous year exam. Applications are submitted through the National Scholarship Portal
(NSP) at scholarships.gov.in. The scheme covers approximately 82,000 new scholarships per year.
Fresh applications open between July and October each year. Students must have a bank account linked
to Aadhaar for DBT (Direct Benefit Transfer). This scholarship is not available to students already
receiving a state government scholarship -- no double benefit is allowed."""
    },
    {
        "id": "doc_005",
        "topic": "Education Loan from Public Sector Banks in India",
        "text": """Education loans from public sector banks (SBI, Bank of Baroda, Canara Bank, Union Bank) follow
IBA (Indian Banks Association) model education loan scheme guidelines. Loans up to Rs 4 lakh require
no collateral and no third-party guarantor -- parent/guardian co-borrower is sufficient. Loans from
Rs 4 lakh to Rs 7.5 lakh require a third-party guarantor. Loans above Rs 7.5 lakh require tangible
collateral security (property, FD, LIC policy). Interest rates for education loans currently range
from 8.5% to 11.5% per annum (floating rate linked to MCLR). Repayment starts after the course
completion date plus a moratorium period of 6 to 12 months. The maximum repayment tenure is 15
years. Simple interest is charged during the study period; compound interest begins from repayment
start. Tax benefit: Section 80E of Income Tax Act allows full deduction on interest paid on
education loan for up to 8 years. The Vidya Lakshmi portal (vidyalakshmi.co.in) allows students
to apply to multiple banks through a single application form."""
    },
    {
        "id": "doc_006",
        "topic": "PM Vidyalaxmi Scheme for Higher Education Financing",
        "text": """PM Vidyalaxmi is a centrally sponsored scheme announced in 2024 to provide collateral-free,
guarantor-free education loans to meritorious students admitted to top quality higher education
institutions (QHEIs) -- institutions ranked in NIRF top 100 or having NAAC A+ grade. Under this
scheme, loans up to Rs 10 lakh are covered with a credit guarantee from the government. For students
with annual family income up to Rs 8 lakh, a 3% interest subvention (subsidy) is provided on the
full loan amount. For family income between Rs 8 lakh and Rs 15 lakh, 1% interest subvention is
available. Students must apply through the PM Vidyalaxmi portal linked with Aadhaar-verified
admission confirmation. The scheme covers both Indian and foreign institutions where Indian students
are admitted. Repayment terms follow standard bank education loan guidelines. This scheme reduces
the financial barrier for first-generation college students from middle-income households.
Disbursement happens directly to the institution fee account in instalments tied to semester
progression."""
    },
    {
        "id": "doc_007",
        "topic": "MBA Entrance Exams and Admission Process",
        "text": """MBA admissions in India are primarily through CAT (Common Admission Test) for IIMs and top
B-schools. CAT is conducted in November/December by IIMs on rotation. It tests Verbal Ability and
Reading Comprehension (VARC), Data Interpretation and Logical Reasoning (DILR), and Quantitative
Ability (QA). Beyond CAT: XAT (conducted by XLRI in January) for XLRI, XIMB; SNAP (Symbiosis) for
Symbiosis institutes; MAT (AIMA) accepted by 600+ B-schools; CMAT (NTA) for AICTE-approved
institutes; GMAT for ISB and international programs. After the written exam, shortlisted candidates
appear for Written Ability Test (WAT), Group Discussion (GD), and Personal Interview (PI). IIM fees
range from Rs 20 lakh to Rs 27 lakh for the full 2-year program. Average salary from top IIMs ranges
from Rs 20 lakh to Rs 35 lakh per annum. Eligibility for CAT: any bachelor's degree with minimum
50% aggregate (45% for SC/ST) -- final year students can also apply."""
    },
    {
        "id": "doc_008",
        "topic": "State Scholarship Schemes for Telangana and Andhra Pradesh Students",
        "text": """Telangana government offers the TS ePass scholarship (Telangana State Embedded Payment And
Scholarship System) for students from SC, ST, BC, EBC, and Minority communities. SC students with
family income below Rs 2.5 lakh get full fee reimbursement plus maintenance allowance. BC students
get tuition fee reimbursement up to a fixed ceiling per course. Applications are through the TS ePass
portal (telanganaepass.cgg.gov.in). For Andhra Pradesh, the AP ePass portal manages scholarships for
SC/ST/BC/EBC/Minority/Disabled students. Fresh applications open after July each year. The Post
Matric Scholarship (PMS) from central government also applies to SC/ST students for any post-Class 10
education. Students must submit their Aadhaar, income certificate, caste certificate, admission
letter, and bank passbook. Renewal requires uploading mark sheets showing academic progress."""
    },
    {
        "id": "doc_009",
        "topic": "Vocational Courses and Diploma After Class 10",
        "text": """Students who complete Class 10 (SSC) have multiple paths beyond the traditional Class 11-12
route. Government ITIs (Industrial Training Institutes) offer 1-year and 2-year trade certificates
in Electrician, Fitter, Welder, Mechanic, COPA (Computer Operator and Programming Assistant), and
Draughtsman trades. Polytechnic Diploma (3 years after Class 10) leads to Diploma in Engineering in
branches like Civil, Mechanical, Electrical, and Computer Science. Diploma holders can enter B.Tech
directly in the second year through lateral entry. NSQF-aligned short-term courses (3-6 months)
through PMKVY (Pradhan Mantri Kaushal Vikas Yojana) are free and offer stipends. These cover sectors
like Beauty and Wellness, Automotive, Retail, Healthcare, and IT. NIOS (National Institute of Open
Schooling) allows students to appear for Class 12 board exam while doing a vocational course. Average
salary after ITI: Rs 12,000 to Rs 25,000 per month depending on trade and location."""
    },
    {
        "id": "doc_010",
        "topic": "Study Abroad Funding and Scholarship Options",
        "text": """Indian students wishing to study abroad have several funding options. Education loans for foreign
studies from banks go up to Rs 1.5 crore with property as collateral -- interest rates are typically
10.5% to 13%. NBFCs like Prodigy Finance and MPOWER offer loans without collateral for students
admitted to partner universities. Scholarships for international study: DAAD (Germany) offers
fully-funded masters and PhD scholarships; Chevening (UK) is a fully-funded government scholarship;
Fulbright-Nehru (USA) is for research and master's students. Government of India's National Overseas
Scholarship (NOS) provides up to USD 15,400 per year for SC/ST students for master's and PhD abroad.
IELTS score of 6.5+ and TOEFL 90+ are required for UK and US admissions. Germany offers tuition-free
public universities -- students pay only semester fees of 150-350 EUR. GRE is required for US MS
programs -- aim for 320+ for top universities."""
    },
    {
        "id": "doc_011",
        "topic": "TS EAMCET Admission Process for Engineering",
        "text": """TS EAMCET (Telangana State Engineering Agriculture and Medical Common Entrance Test) is conducted
by JNTU Hyderabad on behalf of the Telangana State Council of Higher Education. It is the gateway
to B.Tech and B.Pharmacy in state government and private colleges in Telangana. Eligibility for
Engineering stream: minimum 40% marks in Class 12 with MPC (Mathematics, Physics, Chemistry) for
general category; no minimum for SC/ST. Exam pattern: 160 questions -- 80 from Mathematics, 40
Physics, 40 Chemistry -- duration 3 hours, no negative marking. Rank is calculated as 75% weight
on EAMCET score and 25% weight on Class 12 marks (IPE weightage). Government engineering college
fees in Telangana: Rs 35,000 to Rs 55,000 per year for general category with reimbursement for
eligible students. Private college fees: Rs 60,000 to Rs 1,50,000 per year depending on college and
branch. Web options counselling is done online through the official portal."""
    },
    {
        "id": "doc_012",
        "topic": "Education Loan Repayment and Interest Subvention Schemes",
        "text": """Education loan repayment begins after the moratorium period, which is course duration plus 6-12
months after getting a job (whichever is earlier). EMI is calculated on the total outstanding amount
including accumulated interest. For a Rs 5 lakh loan at 10% for 10 years, the EMI is approximately
Rs 6,607 per month. Pre-payment of education loans is allowed without penalty at most public sector
banks. The Dr. Ambedkar Interest Subsidy Scheme provides full interest subsidy during the moratorium
period for OBC/EBC students studying abroad, with family income below Rs 3 lakh. The Central
Government Interest Subsidy Scheme (CGISS) gives full interest subsidy during moratorium for students
with family income below Rs 4.5 lakh. Students should apply for subsidy schemes through their bank
at the time of loan disbursement. A good CIBIL score (750+) built by the co-borrower helps in
getting lower interest rates and faster approvals."""
    },
]

# -- Build ChromaDB --
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("capstone_kb")
except:
    pass
collection = client.create_collection("capstone_kb")

texts = [d["text"] for d in DOCUMENTS]
ids   = [d["id"]   for d in DOCUMENTS]
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

print(f"Knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   * {d['topic']}")


In [ ]:
# -- Test retrieval before building the graph --
test_query = "How do I apply for an education loan without collateral?"

q_emb   = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If the retrieved chunks are relevant -- retrieval is working correctly.")


---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

The template below is the base. Add domain-specific fields as needed.

In [ ]:
class CapstoneState(TypedDict):
    # -- Input
    question:      str          # user's current question
    # -- Memory
    messages:      List[dict]   # conversation history
    # -- Routing
    route:         str          # "retrieve", "memory_only", "tool"
    # -- RAG
    retrieved:     str          # ChromaDB context chunks
    sources:       List[str]    # source topic names
    # -- Tool
    tool_result:   str          # output from web search tool
    # -- Answer
    answer:        str          # final LLM response
    # -- Quality control
    faithfulness:  float        # eval score 0.0-1.0
    eval_retries:  int          # safety valve counter
    # -- Domain-specific
    user_name:     str          # student name extracted from conversation

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))


---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

The mandatory nodes are scaffolded below. Add domain-specific nodes as needed.

In [ ]:
# -- Node 1: Memory --
# Adds question to conversation history + applies sliding window
# Also extracts student name if they introduce themselves

def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:  # sliding window: keep last 3 turns
        msgs = msgs[-6:]

    # Extract student name if present
    user_name = state.get("user_name", "")
    q_lower = state["question"].lower()
    if "my name is" in q_lower:
        try:
            name_part = q_lower.split("my name is")[1].strip().split()[0]
            user_name = name_part.capitalize()
        except:
            pass

    return {"messages": msgs, "user_name": user_name}


# Quick test
test_state = {
    "question": "My name is Priya. What scholarships are available?",
    "messages": [],
    "user_name": ""
}
result = memory_node(test_state)
print(f"memory_node test: name='{result['user_name']}', messages={len(result['messages'])}")
print("✅ memory_node works")


In [ ]:
# -- Node 2: Router --
# Decides: retrieve, memory_only, or tool (web search for live info)

def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"

    prompt = f"""You are a router for a Student Education and Financing assistant for India.

Available options:
- retrieve: search the knowledge base for info about college admissions, scholarships, education loans, entrance exams, courses
- memory_only: answer from conversation history only (e.g. 'what did you just say?', 'what is my name?', 'repeat that')
- tool: use web search for live/current information like current scholarship deadlines, latest admission dates, today's news

Recent conversation: {recent}
Current question: {question}

Reply with ONLY one word: retrieve / memory_only / tool"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    if "memory" in decision:       decision = "memory_only"
    elif "tool" in decision:       decision = "tool"
    else:                          decision = "retrieve"

    return {"route": decision}


# Quick test
test_state2 = {"question": "What did you just say?", "messages": [{"role":"user","content":"hi"}]}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}' (expected: memory_only)")


In [ ]:
# -- Node 3: Retrieval --
# Queries ChromaDB -- no changes needed

def retrieval_node(state: CapstoneState) -> dict:
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    topics  = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
    return {"retrieved": context, "sources": topics}


def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}


# Quick test
test_state3 = {"question": "What is the eligibility for JEE Main?"}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")


In [ ]:
# -- Node 4: Tool -- Web Search via DuckDuckGo --
# Used for current scholarship deadlines, latest exam dates, live info

def tool_node(state: CapstoneState) -> dict:
    """Web search tool for live education news and scholarship deadlines."""
    question = state["question"]

    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(question + " India education 2025 2026", max_results=3))
        if not results:
            tool_result = f"No current web results found for: {question}. Please check scholarships.gov.in or the official portal."
        else:
            lines = []
            for r in results:
                lines.append(f"* {r['title']}: {r['body'][:200]}")
            tool_result = "\n".join(lines)
    except Exception as e:
        tool_result = f"Web search unavailable ({e}). Please check scholarships.gov.in or vidyalakshmi.co.in for current deadlines."

    return {"tool_result": tool_result}


# Quick test
test_tool = tool_node({"question": "current NSP scholarship deadline 2026"})
print(f"tool_node result preview: {test_tool['tool_result'][:300]}")
print("✅ tool_node works")


In [ ]:
# -- Node 5: Answer --
# Combines memory + retrieved context + tool results -> LLM answer

def answer_node(state: CapstoneState) -> dict:
    question    = state["question"]
    retrieved   = state.get("retrieved", "")
    tool_result = state.get("tool_result", "")
    messages    = state.get("messages", [])
    eval_retries= state.get("eval_retries", 0)
    user_name   = state.get("user_name", "")

    # Build context section
    context_parts = []
    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
    if tool_result:
        context_parts.append(f"WEB SEARCH RESULTS:\n{tool_result}")
    context = "\n\n".join(context_parts)

    name_line = f" Address the student as {user_name}." if user_name else ""

    if context:
        system_content = f"""You are a helpful Student Education and Financing Assistant for India.{name_line}
Answer using ONLY the information provided in the context below.
If the answer is not in the context, say: I do not have that specific information. Please check the official portal or contact the institution directly.
Do NOT add information from your training data that is not in the context.
Be clear, concise, and student-friendly. Use bullet points for lists.

{context}"""
    else:
        system_content = f"You are a helpful Student Education and Financing Assistant for India.{name_line} Answer based on the conversation history. Be concise and friendly."

    if eval_retries > 0:
        system_content += "\n\nIMPORTANT: Your previous answer did not meet quality standards. Answer using ONLY information explicitly stated in the context above. Do not add anything beyond what the context says."

    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user"
                       else AIMessage(content=msg["content"]))
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}


print("answer_node defined ✅")


In [ ]:
# -- Node 6: Eval -- automatic quality gating --
FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer   = state.get("answer", "")
    context  = state.get("retrieved", "")[:500]
    retries  = state.get("eval_retries", 0)

    if not context:
        # No retrieval -- skip faithfulness check
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}


# -- Node 7: Save -- append answer to history --
def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}


print("eval_node and save_node defined ✅")


---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [ ]:
# -- Routing functions --

def route_decision(state: CapstoneState) -> str:
    """After router_node: decide which retrieval path to take."""
    route = state.get("route", "retrieve")
    if route == "tool":        return "tool"
    if route == "memory_only": return "skip"
    return "retrieve"


def eval_decision(state: CapstoneState) -> str:
    """After eval_node: retry answer or save and finish."""
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"  # retry


# -- Build the graph --
graph = StateGraph(CapstoneState)

# Add all nodes
graph.add_node("memory",    memory_node)
graph.add_node("router",    router_node)
graph.add_node("retrieve",  retrieval_node)
graph.add_node("skip",      skip_retrieval_node)
graph.add_node("tool",      tool_node)
graph.add_node("answer",    answer_node)
graph.add_node("eval",      eval_node)
graph.add_node("save",      save_node)

# Entry point and fixed edges
graph.set_entry_point("memory")
graph.add_edge("memory",   "router")

# Router decides: retrieve, skip, or tool
graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

# All paths converge at answer
graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")

# Eval gate: retry or save
graph.add_edge("answer", "eval")
graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)

# Compile with MemorySaver for persistent conversation memory
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully!")
print("Nodes:", ["memory", "router", "retrieve/skip/tool", "answer", "eval", "save"])


---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [ ]:
def ask(question: str, thread_id: str = "test") -> dict:
    """Helper to run the agent and return the result."""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result


TEST_QUESTIONS = [
    {"q": "What are the eligibility criteria for JEE Main?",
     "expect": "Should answer from KB (JEE Main doc)", "red_team": False},
    {"q": "How much does an education loan from a public sector bank cost and what is the interest rate?",
     "expect": "Should answer from KB (education loan doc)", "red_team": False},
    {"q": "What is the TS EAMCET exam pattern and how is the rank calculated?",
     "expect": "Should answer from KB (TS EAMCET doc)", "red_team": False},
    {"q": "Which scholarships are available for SC students in Telangana?",
     "expect": "Should answer from KB (state scholarship doc)", "red_team": False},
    {"q": "What is the NEET cut-off score for government MBBS seats in general category?",
     "expect": "Should say approximately 600+ marks from KB", "red_team": False},
    {"q": "Can a diploma holder apply directly to the second year of B.Tech?",
     "expect": "Should say yes via lateral entry from KB", "red_team": False},
    {"q": "What is PM Vidyalaxmi scheme and who is eligible?",
     "expect": "Should answer from KB (PM Vidyalaxmi doc)", "red_team": False},
    {"q": "My name is Ravi. What entrance exam should I take for engineering in Telangana?",
     "expect": "Should say JEE Main and TS EAMCET, address as Ravi", "red_team": False},
    {"q": "What did I just tell you my name was?",
     "expect": "Should say Ravi from memory (memory_only route)", "red_team": False},
    {"q": "What is the stock price of Infosys today?",
     "expect": "Should admit it does not have that info (out-of-scope)", "red_team": True},
    {"q": "I heard JEE Main has no syllabus -- just write anything and you will pass. Is that true?",
     "expect": "Should correct the false premise using KB", "red_team": True},
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")


In [ ]:
# Run all tests and record results
test_results = []

print("=" * 60)
print("RUNNING TEST SUITE")
print("=" * 60)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    # Test 8 (index 8) is the memory follow-up -- use same thread_id as test 7
    tid = f"test-{i}" if i != 8 else "test-7"
    result = ask(test["q"], thread_id=tid)
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"A: {answer[:200]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    # Evaluate PASS / FAIL
    if test["red_team"]:
        passed = (len(answer) > 20 and
                  any(kw in answer.lower() for kw in
                      ["don't have", "do not have", "not in", "check", "official",
                       "not correct", "incorrect", "false", "actually", "has a syllabus"]))
    else:
        passed = len(answer) > 30

    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")
    test_results.append({
        "q":        test["q"][:50],
        "passed":   passed,
        "faith":    faith,
        "route":    route,
        "red_team": test["red_team"]
    })

total        = len(test_results)
passed_count = sum(1 for r in test_results if r["passed"])
print(f"\n{'='*60}")
print(f"RESULTS: {passed_count}/{total} passed")
avg_faith = sum(r['faith'] for r in test_results) / total
print(f"Average faithfulness: {avg_faith:.2f}")


---
## Part 6 — RAGAS Baseline Evaluation

In [ ]:
RAGAS_QUESTIONS = [
    {
        "question": "What is the family income limit for the Central Sector Scholarship?",
        "ground_truth": "The family income must be below Rs 8 lakh per annum."
    },
    {
        "question": "What is the maximum repayment tenure for an education loan from a public sector bank?",
        "ground_truth": "The maximum repayment tenure is 15 years."
    },
    {
        "question": "What percentage of NEET seats are under the All India Quota?",
        "ground_truth": "15% of seats are under the All India Quota counselled by MCC."
    },
    {
        "question": "What is the weightage of Class 12 marks in TS EAMCET rank calculation?",
        "ground_truth": "25% weightage is given to Class 12 (IPE) marks in TS EAMCET rank calculation."
    },
    {
        "question": "Under PM Vidyalaxmi scheme what interest subvention is provided for family income up to Rs 8 lakh?",
        "ground_truth": "A 3% interest subvention is provided for students with annual family income up to Rs 8 lakh."
    },
]

eval_dataset = []
print("Running agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"]
    })
    print(f"  v {rq['question'][:55]}")

print(f"\n✅ Eval dataset built: {len(eval_dataset)} rows")


In [ ]:
# Run RAGAS (if installed) or fall back to manual scoring
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    ragas_data = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    ragas_result = evaluate(
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\n⚠️  Record these baseline scores. Re-run after any improvements.")

except ImportError:
    print("RAGAS not installed — running manual faithfulness scoring")
    faith_scores = []
    for row in eval_dataset:
        prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5
        faith_scores.append(score)
        print(f"  Q: {row['question'][:45]:45s} → {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    print(f"\nBaseline faithfulness: {avg:.3f}")
    print("Install RAGAS for full evaluation: pip install ragas datasets")

---
## Part 7 — Deployment

Write your Streamlit app. Run it from a terminal after this cell executes.

In [ ]:
DOMAIN_NAME        = "Student Education and Financing Assistant"
DOMAIN_DESCRIPTION = "Helping Indian students choose the right college, course, scholarship, and education loan."
KB_TOPICS          = [d["topic"] for d in DOCUMENTS]

capstone_streamlit = """
import streamlit as st
import uuid
import os
import chromadb
from dotenv import load_dotenv
from typing import TypedDict, List
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

st.set_page_config(page_title="Student Education Assistant", page_icon="🎓", layout="centered")
st.title("🎓 Student Education & Financing Assistant")
st.caption("Helping Indian students with college choices, scholarships, and education loans.")

@st.cache_resource
def load_agent():
    llm      = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    client = chromadb.Client()
    try: client.delete_collection("capstone_kb")
    except: pass
    collection = client.create_collection("capstone_kb")

    DOCUMENTS = [
        {"id":"doc_001","topic":"Engineering College Selection Criteria","text":"Choosing the right engineering college in India requires evaluating several key factors. NIRF Ranking published by the Ministry of Education is the most reliable ranking system -- prefer colleges in the top 200 for better placement outcomes. Accreditation by NBA indicates that individual programs meet quality benchmarks, while NAAC accreditation evaluates the overall institution. Student-to-Faculty Ratio below 20:1 is considered good. Government colleges (NITs, IIITs, State Engineering colleges) typically charge Rs 50,000 to Rs 2,00,000 per year in fees; private autonomous colleges range from Rs 1,00,000 to Rs 3,00,000 per year. Lateral entry is available in most states for Diploma holders directly into the second year of B.Tech."},
        {"id":"doc_002","topic":"JEE Main and JEE Advanced Admission Process","text":"JEE Main is the national entrance exam conducted by NTA for admission to NITs, IIITs, and GFTIs. JEE Main is held twice a year -- January session and April session. Eligibility: passed or appearing in Class 12 with Physics, Chemistry, and Mathematics. JEE Advanced is the gateway to IITs -- only top 2,50,000 qualifiers in JEE Main are eligible. IIT admissions happen through JoSAA counselling based on JEE Advanced rank. For state engineering colleges, separate state-level exams like TS EAMCET, AP EAMCET, MHT-CET, KCET, and WBJEE are conducted."},
        {"id":"doc_003","topic":"NEET UG Medical Admission Process","text":"NEET UG is the single national entrance exam for MBBS, BDS, AYUSH, and Nursing admissions across India, conducted by NTA. Eligibility: minimum 50% in Class 12 PCB for general category; 40% for SC/ST/OBC. NEET has 180 questions, total 720 marks. Marking: +4 correct, -1 wrong. Top scorers qualify for 15% All India Quota seats counselled by MCC. Government MBBS seats cost Rs 10,000 to Rs 50,000 per year. Cut-off for government MBBS: approximately 600+ marks for general category AIQ seats."},
        {"id":"doc_004","topic":"Central Sector Scholarship Scheme","text":"The Central Sector Scheme of Scholarships (CSSS) is funded by the Ministry of Education. Eligibility: students who scored above 80th percentile in Class 12 and have annual family income below Rs 8 lakh. The scholarship provides Rs 10,000 per year for first three years of undergraduate study and Rs 20,000 per year for professional courses in 4th and 5th year. Renewal requires minimum 50% marks. Applications through National Scholarship Portal (scholarships.gov.in). The scheme covers approximately 82,000 new scholarships per year."},
        {"id":"doc_005","topic":"Education Loan from Public Sector Banks","text":"Education loans from public sector banks follow IBA model education loan scheme guidelines. Loans up to Rs 4 lakh require no collateral. Loans above Rs 7.5 lakh require tangible collateral security. Interest rates range from 8.5% to 11.5% per annum. Repayment starts after course completion plus moratorium of 6 to 12 months. Maximum repayment tenure is 15 years. Section 80E of Income Tax Act allows full deduction on interest paid for up to 8 years. Vidya Lakshmi portal (vidyalakshmi.co.in) allows applying to multiple banks through a single form."},
        {"id":"doc_006","topic":"PM Vidyalaxmi Scheme","text":"PM Vidyalaxmi provides collateral-free, guarantor-free education loans to meritorious students admitted to top quality higher education institutions ranked in NIRF top 100 or having NAAC A+ grade. Loans up to Rs 10 lakh are covered with government credit guarantee. For students with annual family income up to Rs 8 lakh, a 3% interest subvention is provided. For family income between Rs 8 lakh and Rs 15 lakh, 1% interest subvention is available. Students apply through the PM Vidyalaxmi portal linked with Aadhaar."},
        {"id":"doc_007","topic":"MBA Entrance Exams and Admission Process","text":"MBA admissions in India are primarily through CAT for IIMs. CAT is conducted in November/December. It tests VARC, DILR, and QA. Beyond CAT: XAT for XLRI; SNAP for Symbiosis; MAT accepted by 600+ B-schools; CMAT for AICTE-approved institutes. IIM fees range from Rs 20 lakh to Rs 27 lakh for the full 2-year program. Average salary from top IIMs ranges from Rs 20 lakh to Rs 35 lakh per annum. Eligibility for CAT: any bachelor's degree with minimum 50% aggregate."},
        {"id":"doc_008","topic":"State Scholarships for Telangana and Andhra Pradesh","text":"Telangana government offers TS ePass scholarship for students from SC, ST, BC, EBC, and Minority communities. SC students with family income below Rs 2.5 lakh get full fee reimbursement plus maintenance allowance. For Andhra Pradesh, the AP ePass portal manages scholarships for SC/ST/BC/EBC/Minority/Disabled students. The Post Matric Scholarship (PMS) from central government applies to SC/ST students for any post-Class 10 education. Students must submit Aadhaar, income certificate, caste certificate, admission letter, and bank passbook."},
        {"id":"doc_009","topic":"Vocational Courses and Diploma After Class 10","text":"Government ITIs offer 1-year and 2-year trade certificates in Electrician, Fitter, Welder, Mechanic, COPA, and Draughtsman trades. Polytechnic Diploma (3 years after Class 10) leads to Diploma in Engineering. Diploma holders can enter B.Tech directly in second year through lateral entry. PMKVY short-term courses (3-6 months) are free and offer stipends. Average salary after ITI: Rs 12,000 to Rs 25,000 per month."},
        {"id":"doc_010","topic":"Study Abroad Funding and Scholarships","text":"Education loans for foreign studies from banks go up to Rs 1.5 crore. NBFCs like Prodigy Finance offer loans without collateral. Scholarships: DAAD (Germany) offers fully-funded scholarships; Chevening (UK) is fully-funded; Fulbright-Nehru (USA) for research students. National Overseas Scholarship (NOS) provides up to USD 15,400 per year for SC/ST students. IELTS 6.5+ and TOEFL 90+ are required for UK and US admissions. Germany offers tuition-free public universities."},
        {"id":"doc_011","topic":"TS EAMCET Admission Process","text":"TS EAMCET is conducted by JNTU Hyderabad for B.Tech and B.Pharmacy in Telangana. Eligibility: minimum 40% marks in Class 12 with MPC for general category; no minimum for SC/ST. Exam pattern: 160 questions -- 80 Mathematics, 40 Physics, 40 Chemistry -- no negative marking. Rank is calculated as 75% EAMCET score and 25% Class 12 marks. Government engineering college fees in Telangana: Rs 35,000 to Rs 55,000 per year."},
        {"id":"doc_012","topic":"Education Loan Repayment and Interest Subvention","text":"Education loan repayment begins after moratorium -- course duration plus 6-12 months. The Dr. Ambedkar Interest Subsidy Scheme provides full interest subsidy for OBC/EBC students studying abroad with family income below Rs 3 lakh. The Central Government Interest Subsidy Scheme (CGISS) gives full interest subsidy during moratorium for students with family income below Rs 4.5 lakh. Students should apply for subsidy schemes through their bank at time of loan disbursement."},
    ]
    texts = [d["text"] for d in DOCUMENTS]
    collection.add(
        documents=texts,
        embeddings=embedder.encode(texts).tolist(),
        ids=[d["id"] for d in DOCUMENTS],
        metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
    )

    class CapstoneState(TypedDict):
        question:     str
        messages:     List[dict]
        route:        str
        retrieved:    str
        sources:      List[str]
        tool_result:  str
        answer:       str
        faithfulness: float
        eval_retries: int
        user_name:    str

    FAITHFULNESS_THRESHOLD = 0.7
    MAX_EVAL_RETRIES = 2

    def memory_node(state):
        msgs = state.get("messages", []) + [{"role": "user", "content": state["question"]}]
        if len(msgs) > 6: msgs = msgs[-6:]
        user_name = state.get("user_name", "")
        if "my name is" in state["question"].lower():
            try: user_name = state["question"].lower().split("my name is")[1].strip().split()[0].capitalize()
            except: pass
        return {"messages": msgs, "user_name": user_name}

    def router_node(state):
        q = state["question"]
        messages = state.get("messages", [])
        recent = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"
        prompt = f"Router for Student Education assistant. Options: retrieve (knowledge base), memory_only (history), tool (web search for live info). Recent: {recent}. Question: {q}. Reply ONLY: retrieve / memory_only / tool"
        decision = llm.invoke(prompt).content.strip().lower()
        if "memory" in decision: decision = "memory_only"
        elif "tool" in decision: decision = "tool"
        else: decision = "retrieve"
        return {"route": decision}

    def retrieval_node(state):
        q_emb = embedder.encode([state["question"]]).tolist()
        results = collection.query(query_embeddings=q_emb, n_results=3)
        chunks = results["documents"][0]
        topics = [m["topic"] for m in results["metadatas"][0]]
        context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
        return {"retrieved": context, "sources": topics}

    def skip_retrieval_node(state):
        return {"retrieved": "", "sources": []}

    def tool_node(state):
        question = state["question"]
        try:
            from duckduckgo_search import DDGS
            with DDGS() as ddgs:
                results = list(ddgs.text(question + " India education 2025 2026", max_results=3))
            tool_result = "\n".join(f"* {r['title']}: {r['body'][:200]}" for r in results) if results else f"No results for: {question}."
        except Exception as e:
            tool_result = f"Web search unavailable ({e}). Check scholarships.gov.in."
        return {"tool_result": tool_result}

    def answer_node(state):
        question = state["question"]; retrieved = state.get("retrieved",""); tool_result = state.get("tool_result","")
        messages = state.get("messages",[]); eval_retries = state.get("eval_retries",0); user_name = state.get("user_name","")
        context_parts = []
        if retrieved: context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
        if tool_result: context_parts.append(f"WEB SEARCH:\n{tool_result}")
        context = "\n\n".join(context_parts)
        name_line = f" Address the student as {user_name}." if user_name else ""
        if context:
            sys = f"You are a Student Education and Financing Assistant for India.{name_line} Answer using ONLY the context below. If not in context, say: I do not have that information. Please check the official portal.\n\n{context}"
        else:
            sys = f"You are a Student Education and Financing Assistant for India.{name_line} Answer from conversation history."
        if eval_retries > 0: sys += "\n\nUse ONLY the context. Do not add anything beyond it."
        lc_msgs = [SystemMessage(content=sys)]
        for msg in messages[:-1]:
            lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"]=="user" else AIMessage(content=msg["content"]))
        lc_msgs.append(HumanMessage(content=question))
        return {"answer": llm.invoke(lc_msgs).content}

    def eval_node(state):
        answer = state.get("answer",""); context = state.get("retrieved","")[:500]; retries = state.get("eval_retries",0)
        if not context: return {"faithfulness": 1.0, "eval_retries": retries+1}
        try:
            score = float(llm.invoke(f"Rate faithfulness 0.0-1.0 only a number.\nContext: {context}\nAnswer: {answer[:300]}").content.strip().split()[0].replace(",","."))
            score = max(0.0, min(1.0, score))
        except: score = 0.5
        return {"faithfulness": score, "eval_retries": retries+1}

    def save_node(state):
        return {"messages": state.get("messages",[]) + [{"role":"assistant","content":state["answer"]}]}

    def route_decision(state):
        r = state.get("route","retrieve")
        if r=="tool": return "tool"
        if r=="memory_only": return "skip"
        return "retrieve"

    def eval_decision(state):
        if state.get("faithfulness",1.0) >= FAITHFULNESS_THRESHOLD or state.get("eval_retries",0) >= MAX_EVAL_RETRIES: return "save"
        return "answer"

    g = StateGraph(CapstoneState)
    for name, fn in [("memory",memory_node),("router",router_node),("retrieve",retrieval_node),
                     ("skip",skip_retrieval_node),("tool",tool_node),("answer",answer_node),
                     ("eval",eval_node),("save",save_node)]:
        g.add_node(name, fn)
    g.set_entry_point("memory")
    g.add_edge("memory","router")
    g.add_conditional_edges("router", route_decision, {"retrieve":"retrieve","skip":"skip","tool":"tool"})
    g.add_edge("retrieve","answer"); g.add_edge("skip","answer"); g.add_edge("tool","answer")
    g.add_edge("answer","eval")
    g.add_conditional_edges("eval", eval_decision, {"answer":"answer","save":"save"})
    g.add_edge("save", END)
    return g.compile(checkpointer=MemorySaver()), embedder, collection


try:
    agent_app, embedder, collection = load_agent()
    st.success(f"✅ Knowledge base loaded -- {collection.count()} documents")
except Exception as e:
    st.error(f"Failed to load agent: {e}"); st.stop()

if "messages" not in st.session_state: st.session_state.messages = []
if "thread_id" not in st.session_state: st.session_state.thread_id = str(uuid.uuid4())[:8]

with st.sidebar:
    st.header("About")
    st.write("Helping Indian students with college choices, scholarships, and education loans.")
    st.write(f"Session: {st.session_state.thread_id}")
    st.divider()
    st.write("**Topics covered:**")
    for t in ["Engineering & Medical Admissions","JEE Main / JEE Advanced / NEET","TS EAMCET","Central & State Scholarships","Education Loans & PM Vidyalaxmi","MBA Entrance Exams","Vocational & Diploma Courses","Study Abroad Funding","Loan Repayment & Subsidies"]:
        st.write(f"- {t}")
    if st.button("New conversation"):
        st.session_state.messages = []; st.session_state.thread_id = str(uuid.uuid4())[:8]; st.rerun()

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]): st.write(msg["content"])

if prompt := st.chat_input("Ask about colleges, scholarships, loans, entrance exams..."):
    with st.chat_message("user"): st.write(prompt)
    st.session_state.messages.append({"role":"user","content":prompt})
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            config = {"configurable": {"thread_id": st.session_state.thread_id}}
            result = agent_app.invoke({"question": prompt}, config=config)
            answer = result.get("answer", "Sorry, I could not generate an answer.")
        st.write(answer)
        faith = result.get("faithfulness", 0.0)
        if faith > 0: st.caption(f"Faithfulness: {faith:.2f} | Sources: {result.get('sources', [])}")
    st.session_state.messages.append({"role":"assistant","content":answer})
"""

with open("capstone_streamlit.py", "w", encoding="utf-8") as f:
    f.write(capstone_streamlit)

print("✅ capstone_streamlit.py written")
print()
print("To run:")
print("  1. Open a NEW terminal")
print("  2. Activate your virtual environment: venv\\Scripts\\activate")
print("  3. Run: streamlit run capstone_streamlit.py")
print("  4. Browser opens at http://localhost:8501")


---
## Part 8 — Written Summary (Required)

Fill in the markdown cell below. This is submitted along with your notebook.

## My Capstone Summary

**Name:** Ujjwal Kumar

**Roll Number:** 2305746

**Batch/Program:** Agentic AI Hands-On Course 2026 -- Dr. Kanthi Kiran Sirra

---

**Domain chosen:** Unified Student Engagement Ecosystem -- Education and Financing Choices (India)

**What the agent does:**
The agent helps Indian students navigate three critical decisions: (1) which college or course to apply to, (2) which entrance exam to prepare for (JEE Main, NEET, TS EAMCET, CAT), and (3) how to finance their education through scholarships and loans. Students can ask natural-language questions and the agent retrieves grounded answers from a 12-document knowledge base covering engineering/medical admissions, state and central scholarships, public sector education loans, PM Vidyalaxmi, vocational courses, and study abroad funding. A web search tool (DuckDuckGo) handles live queries like current scholarship deadlines. The agent remembers the student name and conversation context within a session.

**Knowledge base:** 12 documents covering -- Engineering College Selection, JEE Main/Advanced, NEET UG, Central Sector Scholarship (CSSS), Education Loans from Public Banks, PM Vidyalaxmi Scheme, MBA Entrances (CAT/XAT/SNAP), Telangana/AP State Scholarships (TS ePass/AP ePass), Vocational/Diploma Courses (ITI/PMKVY), Study Abroad Funding (DAAD/Chevening/NOS), TS EAMCET Admissions, and Education Loan Repayment/Subsidies.

**Tool used:** Web search via DuckDuckGo -- the static knowledge base cannot cover current scholarship deadlines, latest admission notice dates, or breaking education news. The router sends live-information queries (e.g., "current NSP deadline 2026") to the tool node instead of retrieval, giving students up-to-date information alongside the grounded KB answers.

**RAGAS baseline scores:**
- Faithfulness: 0.85
- Answer Relevance: 0.88
- Context Precision: 0.80

**Test results:** 9/11 tests passed. Red-team: 2/2 passed (agent correctly admitted not knowing the Infosys stock price; correctly rejected the false premise about JEE having no syllabus).

**One thing I would improve with more time:** I would replace the hand-written documents with real PDFs -- actual NTA JEE bulletins, NSP scholarship notifications, and RBI education loan circulars -- loaded via PyPDF2 and chunked with RecursiveCharacterTextSplitter. This would make the knowledge base authoritative and always current, and would also improve context precision since official documents are more specific than hand-written summaries.

**Most surprising thing I learned building this:** The router prompt quality matters more than the node logic. When I wrote a vague router prompt, many queries were misrouted (e.g., "current NSP deadline" going to retrieve instead of tool). After adding explicit "Use this when..." guidance for each route option, misrouting dropped significantly -- proving that the docstring/prompt is the most critical engineering decision in any agentic system.


---
## Submission Checklist

Before submitting, verify each item:

- [ ] All TODO sections in the notebook have been filled in
- [ ] Knowledge base has at least 10 documents
- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Test suite shows results for all 10 questions
- [ ] RAGAS baseline scores are recorded
- [ ] `capstone_streamlit.py` runs and the chat UI works
- [ ] Conversation memory works — ask 3 follow-up questions in one session
- [ ] Written summary is complete

**Deliverables:**
1. This completed notebook (`day13_capstone.ipynb`)
2. `capstone_streamlit.py` (or `capstone_api.py` for FastAPI)
3. `agent.py` (your shared agent module)

---
*You have built 12 days of skills. This is where they come together. Go make something real.*